# A Game of Thrones — AlphaZero training on Google Colab

This notebook clones the repository, installs the wheel, runs a smoke gate and
then launches a **resumable** run into `run4`. Checkpoints are written to Google
Drive, so a runtime that disconnects can continue from the last iteration.

`run3` predates the sequential Westeros resolution and the PIMC deck resampling;
it is a pre-change baseline, not something to resume. The wheel is cached on
Drive by source tree + platform + Python, so a docs-only commit reuses it while a
Rust/Python/data change rebuilds it.

**Before you start**

1. `Runtime > Change runtime type`: a **CPU** runtime is fine. The search is
   CPU-bound, so a GPU adds little.
2. The GitHub repository is **private**. Create a fine-grained read-only token
   for it and add it in the 🔑 panel (left sidebar) as a secret named
   `GITHUB_TOKEN`. If the repository is public, skip this.
3. Keep this tab open while training runs. If Colab disconnects you, the next
   session resumes from the last checkpoint on Drive.


In [ ]:
import os
import sys

import torch

affinity = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else None
print("python:", sys.version.split()[0])
print("torch:", torch.__version__, "| cuda available:", torch.cuda.is_available())
print("usable cpus:", affinity or os.cpu_count())

In [ ]:
%cd /content
import os
import subprocess
from pathlib import Path

REPO_DIR = Path("/content/game-of-thrones")
if not REPO_DIR.exists():
    try:
        from google.colab import userdata

        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = None
    url = "https://github.com/canditoo2/game-of-thrones.git"
    if token:
        url = f"https://{token}@github.com/canditoo2/game-of-thrones.git"
    subprocess.run(
        ["git", "clone", "--depth", "1", url, str(REPO_DIR)],
        check=True,
    )

# A failed pull must stop the notebook: otherwise Colab would install the old
# commit's cached wheel and run4 would train stale code while looking healthy.
subprocess.run(
    ["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", "main"],
    check=True,
)
subprocess.run(
    ["git", "-C", str(REPO_DIR), "reset", "--hard", "FETCH_HEAD"],
    check=True,
)
head = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()
fetched = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "FETCH_HEAD"], text=True
).strip()
assert head == fetched, (head, fetched)
subject = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "log", "-1", "--format=%s"], text=True
).strip()
print("training commit", head[:12], "|", subject)
os.chdir(REPO_DIR)


## 1. Mount Drive and install the engine

The engine is a Rust extension. The wheel cache key is the **source tree**
(`rust/`, `src/`, `pyproject.toml`) plus platform and Python, so changing only
documentation does not rebuild it. `RUN_DIR` is `run4`; `run3` is obsolete.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

import glob
import hashlib
import json
import os
import platform
import subprocess
import sys
import zipfile
from pathlib import Path

REPO_DIR = Path("/content/game-of-thrones")
RUN_ROOT = Path("/content/drive/MyDrive/agot")
RUN_DIR = RUN_ROOT / "run4"
ITERATIONS = 60  # total across sessions; resume picks up from the state file
RUN_DIR.mkdir(parents=True, exist_ok=True)

commit = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()
source_tree = subprocess.check_output(
    [
        "git",
        "-C",
        str(REPO_DIR),
        "rev-parse",
        "HEAD:rust",
        "HEAD:src",
        "HEAD:pyproject.toml",
    ],
    text=True,
).strip()
source_key = hashlib.sha1(source_tree.encode()).hexdigest()[:12]
wheel_key = (
    f"{source_key}-{sys.platform}-{platform.machine()}-"
    f"{sys.version_info.major}{sys.version_info.minor}"
)
wheel_dir = RUN_ROOT / "wheels" / wheel_key
wheel_dir.mkdir(parents=True, exist_ok=True)


def usable_wheels():
    paths = sorted(glob.glob(str(wheel_dir / "agot-*.whl")))
    valid = [path for path in paths if zipfile.is_zipfile(path)]
    if sys.platform.startswith("linux"):
        tags = ("manylinux", "linux")
    elif sys.platform == "win32":
        tags = ("win",)
    elif sys.platform == "darwin":
        tags = ("macosx",)
    else:
        tags = (sys.platform,)
    return [
        path
        for path in valid
        if any(tag in os.path.basename(path) for tag in tags)
    ]


wheels = usable_wheels()
if wheels:
    print("using cached wheel", wheels[0])
else:
    print("building the Rust wheel for", wheel_key)
    !curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
    os.environ["PATH"] = (
        os.path.expanduser("~/.cargo/bin") + os.pathsep + os.environ["PATH"]
    )
    os.environ["MATURIN_PEP517_ARGS"] = "--profile release"
    os.chdir(REPO_DIR)
    !pip wheel . --no-deps --wheel-dir {wheel_dir}
    wheels = usable_wheels()

assert wheels, "the wheel build produced no usable artefact"
wheel = Path(wheels[0])
with zipfile.ZipFile(wheel) as archive:
    names = archive.namelist()
assert any(name.startswith("agot/_core") for name in names), names[:10]
assert "agot/data/map_base_6p.json" in names
!pip install -q --force-reinstall --no-deps {wheel}
!pip install -q pytest pytest-cov ruff

import agot
from agot import _core

installed = Path(agot.__file__).resolve()
assert str(REPO_DIR) not in str(installed), installed
print("agot", agot.__version__, "| installed", installed)
print("engine", _core.backend())
assert _core.backend() == "rust"

manifest = {
    "commit": commit,
    "source_key": source_key,
    "wheel": wheel.name,
    "python": sys.version,
    "platform": sys.platform,
    "machine": platform.machine(),
    "iterations": ITERATIONS,
}
(RUN_DIR / "run.json").write_text(
    json.dumps(manifest, indent=2), encoding="utf-8"
)
print("checkpoints ->", RUN_DIR)


In [ ]:
import agot
from agot import _core
from agot.engine.game import BACKEND, Game, GameConfig
from agot.engine.rust import RustGame

config = GameConfig(seed=0, use_westeros_decks=False)
assert config.sequential_westeros is True
assert "pending_choice" in dir(_core.Game)
game = Game.new(config)
assert isinstance(game, RustGame)
assert _core.backend() == "rust"
assert game.legal_actions()
print(
    f"agot {agot.__version__} | package engine {BACKEND} | "
    f"extension {_core.backend()} | legal actions {len(game.legal_actions())}"
)

!agot demo --policy greedy --seed 1
!python -m pytest tests/test_map.py tests/test_engine.py tests/test_rust_engine.py tests/test_batched_selfplay.py -q


## 2. Smoke gate (a few minutes)

Three short iterations on six games. This checks the real sequential/PIMC
pipeline, the native driver, checkpoint reload and the log — not just that the
process starts. A handful of iterations will not prove learning; a crash, a
frozen loss or a stale wheel will show up here instead of hours into a run.


In [ ]:
!rm -rf checkpoints/smoke
!agot train --iterations 3 --games 6 --workers 2 --batch-games 3 --train-threads 2 --buffer 20000 --eval-games 6 --eval-every 3 --checkpoint checkpoints/smoke/agot.pt


In [ ]:
from pathlib import Path

from agot.nn.model import PolicyValueNet
from agot.training.checkpoint import RunState, state_path
from agot.training.report import load_records

log = Path("checkpoints/smoke/agot.log.jsonl")
records = load_records(log)
assert len(records) == 3, records
for record in records:
    print(
        f"iter {record['iteration']:>3}  "
        f"policy_loss {record['policy_loss']:.4f}  "
        f"value_loss {record['value_loss']:.4f}  "
        f"({record.get('seconds', 0):.0f}s)"
    )
    assert record.get("policy_loss") is not None
    assert record.get("value_loss") is not None

losses = [record["policy_loss"] for record in records]
assert len(set(losses)) > 1, "policy loss never changed across three iterations"
assert losses[-1] < 4.0, losses[-1]
PolicyValueNet.load("checkpoints/smoke/agot.pt")
state = RunState.load(state_path("checkpoints/smoke/agot.pt"))
assert state is not None
assert state.iteration == 3
print("OK: the pipeline ran, the loss moved and the checkpoint reloads.")


## 3. Launch the training run

`run4` starts fresh; `run3` is a pre-change baseline. The target is **60
iterations** of 48 games, spread across sessions: a free 2-vCPU runtime usually
takes roughly 5–15 minutes per iteration depending on the batch and eval, so
re-run this cell after a disconnect and it resumes from `agot.state.pt`.

The periodic eval is 12 games every 20 iterations at the training search budget,
so it measures the same agent self-play searched. Watch the policy-loss **trend**
and the best-checkpoint eval, not only the last checkpoint. With the larger
choice action set the old absolute “~2.0” warning is no longer a hard gate; the
smoke gate checks that the loss actually moves.


In [ ]:
import subprocess

from agot.training.checkpoint import RunState, state_path

checkpoint = RUN_DIR / "agot.pt"
state_file = state_path(checkpoint)
resume = checkpoint.exists() and state_file.exists()
state = RunState.load(state_file) if resume else None
done = state.iteration if state is not None else 0
print("checkpoint:", checkpoint)
print(("resuming at iteration " if state is not None else "starting fresh "), done)

if done >= ITERATIONS:
    print(f"already at {done}/{ITERATIONS}; raise ITERATIONS to continue")
else:
    command = [
        "agot",
        "train",
        "--iterations",
        str(ITERATIONS),
        "--games",
        "48",
        "--workers",
        "2",
        "--batch-games",
        "12",
        "--buffer",
        "80000",
        "--opponent-rate",
        "0.2",
        "--train-threads",
        "2",
        "--temperature-rounds",
        "3",
        "--final-temperature",
        "0.25",
        "--eval-games",
        "12",
        "--eval-every",
        "20",
        "--checkpoint",
        str(checkpoint),
    ]
    if state is not None:
        command += ["--resume", str(checkpoint)]
    subprocess.run(command, check=True)


## 4. Monitor

The training cell prints one line per iteration. This cell reads the JSONL
tolerantly (a disconnect can truncate the last line), prints the last few
iterations and renders the loss/eval curves to `/content/run-report.html`.


In [ ]:
from pathlib import Path

from IPython.display import HTML, display

from agot.training.report import load_records, render_file

log = RUN_DIR / "agot.log.jsonl"
if not log.exists():
    print("no log yet — run the training cell first")
else:
    records = load_records(log)
    for record in records[-5:]:
        rate = record.get("eval")
        eval_text = f"  eval {rate:.3f}" if rate is not None else ""
        print(
            f"iter {record['iteration']:>4}  "
            f"policy_loss {record['policy_loss']:.4f}  "
            f"value_loss {record['value_loss']:.4f}{eval_text}"
        )
    page = render_file(
        log,
        out="/content/run-report.html",
        refresh=0,
        target_iterations=ITERATIONS,
    )
    display(HTML(Path(page).read_text(encoding="utf-8")))


## 5. Evaluate the best checkpoint

`agot gauntlet` puts the saved network in one rotating seat against five
opponents. The explicit `--simulations 128 --determinizations 2` matters: the
CLI's cheap default is 24/1, which would measure a weaker search than the run
trained with. The cells use `agot-best.pt` when it exists and fall back to the
last checkpoint before the first evaluation.


In [ ]:
from pathlib import Path

best = RUN_DIR / "agot-best.pt"
checkpoint = best if best.exists() else RUN_DIR / "agot.pt"
print("evaluating", checkpoint)
!agot gauntlet --candidate net:{checkpoint} --opponents random --games 60 --simulations 128 --determinizations 2


In [ ]:
from pathlib import Path

best = RUN_DIR / "agot-best.pt"
first = best if best.exists() else RUN_DIR / "agot.pt"
!agot round-robin --checkpoints {first} {RUN_DIR}/agot.pt --games 30 --simulations 128 --determinizations 2


## Troubleshooting

- **Out of memory** (a free runtime has ~12 GB): the launch uses an 80k replay
  buffer. Lower it to `--buffer 40000` and/or `--games 32`; the batch is 12
  games per worker, so a smaller `--batch-games` also lowers peak MCTS memory.
- **Disconnected**: re-run the clone/install cell and the launch cell; it
  installs the cached wheel and resumes `run4` from the Drive checkpoint. The
  replay buffer restarts empty by design, so a transient loss spike on the first
  resumed iteration is expected.
- **Stale wheel**: the cache key covers `rust/`, `src/` and `pyproject.toml`;
  `run4/run.json` records the commit and wheel name. If in doubt, delete that
  `wheels/<key>` folder and re-run the install cell.
- **Policy loss flat for many iterations**: read `docs/methods.md` and compare
  the eval trend. The old absolute ~2.0 signature was measured with the smaller
  choice action set; with generic choices, use the trend.
- **`agot: command not found`**: re-run the install cell.
- **You pulled a new commit in the same session**: the extension is compiled
  code and cannot be reloaded in place. Run the install cell, then
  `Runtime > Restart runtime`.
- **More throughput**: keep `--workers 2` and `--batch-games 12` on a 2-vCPU
  runtime; the printed `seconds` per iteration is the number to watch.
